[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [1]:
import torch
import torch.nn as nn
import math

In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)


    def forward(self, x_q, x_kv):
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
        # return: (B, S_q, D) — output

        shape_q = x_q.shape
        shape_kv = x_kv.shape

        assert len(shape_q) == 3 and len(shape_kv) == 3
        assert shape_q[-1] == shape_kv[-1] == self.d_model

        sq = shape_q[1]
        skv = shape_kv[1]

        q = self.W_q(x_q)   # shape: B, s_q,  D
        k = self.W_k(x_kv)  # shape: B, s_kv, D
        v = self.W_v(x_kv)  # shape: B, s_kv, D


        q = q.view(shape_q[0],  sq,  self.num_heads, self.d_k)
        k = k.view(shape_kv[0], skv, self.num_heads, self.d_k)
        v = v.view(shape_kv[0], skv, self.num_heads, self.d_k)

        #q = q.transpose(1,2)
        #k = k.transpose(1,2)
        #v = v.transpose(1,2)

        foo = torch.einsum("bqhk, bvhk -> bhqv", q, k)
        foo = nn.functional.softmax(foo / math.sqrt(self.d_k), dim=-1)
        foo = torch.einsum("bhqv, bvhk -> bhqk", foo, v)

        return foo.view(shape_q[0], sq, self.d_model)
        # Q from x_q, K/V from x_kv, no causal mask

In [16]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.2ms)
  ✅ [2/4] Q and KV different lengths (3.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (3.2ms)
  ✅ [4/4] Gradient flow (2.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (12.3ms total)
  Progress saved. Run status() to see your dashboard.

